<a href="https://colab.research.google.com/github/adithya-prabhu-22/ml_lab_2_4NI23CI007_adithya/blob/main/rag_pipe_line.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U sentence-transformers faiss-cpu chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
documents = [
    "The minimum attendance requirement for students is 85 percent.",
    "Students must pay examination fees before the announced deadline.",
    "The college library closes at 8 PM on working days.",
    "Internal assessment marks are published on the student portal.",
    "Students can contact the department office for timetable questions.",
    "The computer laboratory is available during scheduled practical sessions."
]

for i, document in enumerate(documents, start=1):
    print(f"{i}. {document}")

1. The minimum attendance requirement for students is 85 percent.
2. Students must pay examination fees before the announced deadline.
3. The college library closes at 8 PM on working days.
4. Internal assessment marks are published on the student portal.
5. Students can contact the department office for timetable questions.
6. The computer laboratory is available during scheduled practical sessions.


In [ ]:
doccument_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

In [ ]:
print(doccument_embeddings.shape)

(6, 384)


In [ ]:
query = "what is the required  attendance percentage "

In [ ]:
query_embedding = embedding_model.encode(query)

In [ ]:
print(query_embedding.shape)

(384,)


In [ ]:
dimensions = doccument_embeddings.shape[1]

In [ ]:
import faiss
faiss_index = faiss.IndexFlatL2(dimensions)

In [ ]:
faiss_index.add(doccument_embeddings.astype("float32"))

In [ ]:
k = 2
distances, indices = faiss_index.search(query_embedding.astype("float32").reshape(1, -1), k)

In [ ]:
for rank, (distance, index) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank: {rank+1}, Distance: {distance:.4f}, Document: {documents[index]}")

Rank: 1, Distance: 0.4072, Document: The minimum attendance requirement for students is 85 percent.
Rank: 2, Distance: 1.3050, Document: Students must pay examination fees before the announced deadline.


In [ ]:
test_queries = [
    "When does the library shut?",
    "How can I pay my exam fees?",
    "Where are internal marks displayed?",
    "Who should I ask about the timetable?",
    "When can I use the computer lab?"
]


for query in test_queries:
  query_vector = embedding_model.encode(
      [query],
      convert_to_numpy = True
  ).astype("float32")

  distances, indices = faiss_index.search(query_vector, 1)
  best_index = indices[0][0]

  print("QUERY: ", query)
  print("BEST MATCH: ", documents[best_index])
  print("DISTANCE: ", distances[0][0])
  print("-" * 70)

QUERY:  When does the library shut?
BEST MATCH:  The college library closes at 8 PM on working days.
DISTANCE:  0.57711065
----------------------------------------------------------------------
QUERY:  How can I pay my exam fees?
BEST MATCH:  Students must pay examination fees before the announced deadline.
DISTANCE:  0.6211654
----------------------------------------------------------------------
QUERY:  Where are internal marks displayed?
BEST MATCH:  Internal assessment marks are published on the student portal.
DISTANCE:  0.8032142
----------------------------------------------------------------------
QUERY:  Who should I ask about the timetable?
BEST MATCH:  Students can contact the department office for timetable questions.
DISTANCE:  0.69258153
----------------------------------------------------------------------
QUERY:  When can I use the computer lab?
BEST MATCH:  The computer laboratory is available during scheduled practical sessions.
DISTANCE:  0.5136347
------------------

In [ ]:
import chromadb
chroma_client = chromadb.Client()

metadata_collection = chroma_client.get_or_create_collection(
    name="college_docs_with_metadata"
)

metadata_values = [
    {"category": "attendance"},
    {"category": "fees"},
    {"category": "library"},
    {"category": "assessment"},
    {"category": "administration"},
    {"category": "laboratory"}
]


document_ids = [f"doc_{i}" for i in range(len(documents))]

metadata_collection.add(
    ids=document_ids,
    documents=documents,
    embeddings=doccument_embeddings.tolist(),
    metadatas=metadata_values
)

print("Documents with metadata added.")

Documents with metadata added.


In [ ]:
def semantic_search_faiss(query, top_k=3):
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):
        results.append({
            "document": documents[index],
            "distance": float(distance),
            "index": int(index)
        })

    return results


results = semantic_search_faiss(
    "Where can I find my internal marks?",
    top_k=3
)

for item in results:
    print(item)

{'document': 'Internal assessment marks are published on the student portal.', 'distance': 0.8999649286270142, 'index': 3}
{'document': 'Students can contact the department office for timetable questions.', 'distance': 1.7210479974746704, 'index': 4}
{'document': 'Students must pay examination fees before the announced deadline.', 'distance': 1.729201078414917, 'index': 1}
